In [4]:
import requests
from urllib.parse import quote
from datetime import datetime
from dotenv import load_dotenv
import os

# 🔐 .env 파일 로드
load_dotenv()
SERVICE_KEY = os.getenv("SERVICE_KEY")

if not SERVICE_KEY:
    raise ValueError("❌ .env 파일에서 SERVICE_KEY를 불러오지 못했습니다!")

def fetch_data_from_kma(current_time_kst, page_no, category, fcst_time):
    """
    기상청 초단기 예보 API로 특정 시각, 항목(category)의 데이터를 조회하는 함수
    """
    try:
        base_date = current_time_kst.strftime("%Y%m%d")
        params = {
            'serviceKey': quote(SERVICE_KEY, safe=''),
            'numOfRows': 10,
            'pageNo': page_no,
            'dataType': 'JSON',
            'base_date': base_date,
            'base_time': '0200',  # 새벽 2시 발표 기준
            'nx': 98,  # ✅ 부산시 중구 격자 좌표
            'ny': 75
        }

        url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        items = data['response']['body']['items']['item']
        found = next(filter(lambda x: x['category'] == category and x['fcstTime'] == fcst_time, items), None)
        return found['fcstValue'] if found else None

    except requests.exceptions.RequestException as err:
        print("⚠️ 요청 오류:", err)
        return None


In [5]:
import arrow

STATUS_OF_SKY = {
    '1': '☀️ 맑음',
    '3': '⛅ 구름많음',
    '4': '☁️ 흐림'
}

STATUS_OF_PRECIPITATION = {
    '0': '없음',
    '1': '비',
    '2': '비/눈',
    '3': '눈',
    '4': '소나기'
}

def main():
    # 현재 날짜 (KST 기준)
    current_time_kst = arrow.now('Asia/Seoul')
    date_format = "YYYY년 MM월 DD일 dddd"
    date_of_today = current_time_kst.format(date_format, locale="ko_kr")

    # 날씨 데이터 불러오기
    sky = fetch_data_from_kma(current_time_kst, '3', 'SKY', '0500')
    precipitation = fetch_data_from_kma(current_time_kst, '4', 'PTY', '0500')
    lowest_temp = fetch_data_from_kma(current_time_kst, '5', 'TMN', '0600')
    highest_temp = fetch_data_from_kma(current_time_kst, '16', 'TMX', '1500')

    # 메시지 출력
    if None in (sky, precipitation, lowest_temp, highest_temp):
        print("❌ 날씨 정보를 가져오지 못했습니다. 😢")
    else:
        weather_of_today = f"{STATUS_OF_SKY.get(sky, '정보 없음')} (강수: {STATUS_OF_PRECIPITATION.get(precipitation, '정보 없음')})"
        print(
            f"📅 {date_of_today}\n"
            f"🌏 현재 날씨: {weather_of_today}\n"
            f"🔼 최고 기온: {highest_temp}°C\n"
            f"🔽 최저 기온: {lowest_temp}°C\n"
            f"🔎 관측 지점: 부산시 중구"
        )

# 실행
main()

📅 2025년 11월 06일 목요일
🌏 현재 날씨: ☀️ 맑음 (강수: 없음)
🔼 최고 기온: 21.0°C
🔽 최저 기온: 11.0°C
🔎 관측 지점: 부산시 중구
